# Exploratory Data Analysis: Nero Dataset

**Цель:** Анализ данных для прогнозирования разворотов тренда на Forex (H1)

**Структура данных:**
- X ∈ R^{5042×99×11} — входной тензор
- y ∈ {-1, 0, 1} — целевая переменная (экстремальный дисбаланс классов)
- 11 признаков фрактала: fractal_time, price, direction, front, back, strong, break, reverse, power, count, impulse

In [ ]:
# Импорт библиотек
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import mannwhitneyu, ttest_ind, shapiro, normaltest
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from collections import defaultdict
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Настройки визуализации
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14

# Путь для сохранения графиков
PLOTS_DIR = 'plots/'

# Цвета для классов
CLASS_COLORS = {-1: '#e74c3c', 0: '#3498db', 1: '#2ecc71'}
CLASS_NAMES = {-1: 'Sell (-1)', 0: 'Neutral (0)', 1: 'Buy (1)'}

print("Библиотеки загружены успешно!")

## 1. Загрузка и парсинг данных

In [ ]:
def parse_fractal_string(fractal_str: str) -> dict:
    """
    Парсинг строки фрактала в словарь признаков.
    
    Args:
        fractal_str: строка формата 'time:price:direction:front:back:strong:break:reverse:power:count:impulse'
    
    Returns:
        dict с 11 признаками или None при ошибке парсинга
    """
    parts = str(fractal_str).split(':')
    if len(parts) != 11:
        return None
    
    try:
        return {
            'fractal_time': int(parts[0]),
            'price': float(parts[1]),
            'direction': int(parts[2]),
            'front': float(parts[3]),
            'back': float(parts[4]),
            'strong': int(parts[5]),
            'break': int(parts[6]),
            'reverse': int(parts[7]),
            'power': float(parts[8]),
            'count': int(parts[9]),
            'impulse': float(parts[10])
        }
    except (ValueError, IndexError):
        return None


def load_nero_data(filepath: str) -> tuple:
    """
    Загрузка и парсинг данных Nero.
    
    Args:
        filepath: путь к CSV файлу
    
    Returns:
        tuple: (raw_df, fractal_0_df, all_fractals_dict)
    """
    print(f"Загрузка данных из {filepath}...")
    df = pd.read_csv(filepath, sep=';', low_memory=False)
    df.columns = df.columns.str.strip()
    
    print(f"Размер датасета: {df.shape}")
    print(f"Колонки: {list(df.columns[:5])}... (всего {len(df.columns)})")
    
    # Определяем колонки фракталов
    fractal_cols = sorted([col for col in df.columns if col.startswith('fractal')])
    print(f"Найдено {len(fractal_cols)} фрактальных колонок")
    
    # Парсинг первого фрактала (fractal[0]) для всех строк
    first_fractal_col = fractal_cols[0]
    fractal_0_data = []
    
    for idx, row in df.iterrows():
        parsed = parse_fractal_string(row[first_fractal_col])
        if parsed:
            parsed['signal'] = row['signal']
            parsed['row_idx'] = idx
            fractal_0_data.append(parsed)
    
    fractal_0_df = pd.DataFrame(fractal_0_data)
    print(f"Успешно распарсено {len(fractal_0_df)} строк fractal[0]")
    
    return df, fractal_0_df, fractal_cols

# Загрузка данных
raw_df, fractal_0_df, fractal_cols = load_nero_data('Nero_train_labeled.csv')

# Признаки фракталов
FEATURE_NAMES = ['fractal_time', 'price', 'direction', 'front', 'back', 
                 'strong', 'break', 'reverse', 'power', 'count', 'impulse']

print("\nПервые строки fractal[0]:")
fractal_0_df.head()

In [ ]:
# Распределение классов
class_dist = fractal_0_df['signal'].value_counts().sort_index()
print("Распределение классов:")
for cls, count in class_dist.items():
    pct = count / len(fractal_0_df) * 100
    print(f"  Класс {cls:2d}: {count:5d} ({pct:.2f}%)")

# Визуализация распределения классов
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
colors = [CLASS_COLORS[c] for c in class_dist.index]
axes[0].bar([CLASS_NAMES[c] for c in class_dist.index], class_dist.values, color=colors, edgecolor='black')
axes[0].set_title('Распределение классов (абсолютные значения)')
axes[0].set_ylabel('Количество')
for i, (cls, count) in enumerate(class_dist.items()):
    axes[0].text(i, count + 50, str(count), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(class_dist.values, labels=[CLASS_NAMES[c] for c in class_dist.index], 
            colors=colors, autopct='%1.1f%%', startangle=90, explode=[0.05]*3)
axes[1].set_title('Распределение классов (проценты)')

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n⚠️ ВНИМАНИЕ: Экстремальный дисбаланс классов!")
print(f"   Minority классы: -1 (n={class_dist[-1]}), +1 (n={class_dist[1]})")
print(f"   Majority класс: 0 (n={class_dist[0]}, {class_dist[0]/len(fractal_0_df)*100:.1f}%)")

## 2. Статистический анализ признаков по классам

In [ ]:
def compute_feature_stats_by_class(df: pd.DataFrame, features: list) -> pd.DataFrame:
    """
    Вычисление статистик для каждого признака по классам.
    
    Args:
        df: DataFrame с данными
        features: список признаков для анализа
    
    Returns:
        DataFrame со статистиками
    """
    stats_data = []
    
    for feature in features:
        for signal in [-1, 0, 1]:
            values = df[df['signal'] == signal][feature]
            
            stats_data.append({
                'feature': feature,
                'class': signal,
                'count': len(values),
                'mean': values.mean(),
                'std': values.std(),
                'min': values.min(),
                'q25': values.quantile(0.25),
                'median': values.quantile(0.50),
                'q75': values.quantile(0.75),
                'max': values.max()
            })
    
    return pd.DataFrame(stats_data)

# Исключаем fractal_time для статистического анализа признаков
analysis_features = [f for f in FEATURE_NAMES if f != 'fractal_time']

# Вычисление статистик
stats_df = compute_feature_stats_by_class(fractal_0_df, analysis_features)

# Форматированный вывод
print("=" * 100)
print("СТАТИСТИКА ПРИЗНАКОВ ПО КЛАССАМ (fractal[0])")
print("=" * 100)

for feature in analysis_features:
    print(f"\n📊 {feature.upper()}")
    print("-" * 80)
    feature_stats = stats_df[stats_df['feature'] == feature]
    display_df = feature_stats[['class', 'count', 'mean', 'std', 'min', 'q25', 'median', 'q75', 'max']].copy()
    display_df = display_df.round(4)
    print(display_df.to_string(index=False))

# Сохранение статистик
stats_df.to_csv(f'{PLOTS_DIR}feature_stats_by_class.csv', index=False)
print(f"\n✅ Статистики сохранены в {PLOTS_DIR}feature_stats_by_class.csv")

### 2.1 Гистограммы распределений по классам

In [ ]:
def plot_histograms_by_class(df: pd.DataFrame, features: list, save_path: str):
    """
    Построение гистограмм для каждого признака с разбивкой по классам.
    """
    n_features = len(features)
    n_cols = 3
    n_rows = (n_features + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
    axes = axes.flatten()
    
    for idx, feature in enumerate(features):
        ax = axes[idx]
        
        for signal in [-1, 0, 1]:
            values = df[df['signal'] == signal][feature]
            ax.hist(values, bins=30, alpha=0.5, label=CLASS_NAMES[signal], 
                    color=CLASS_COLORS[signal], edgecolor='black', linewidth=0.5)
        
        ax.set_title(feature, fontweight='bold')
        ax.set_xlabel('Значение')
        ax.set_ylabel('Частота')
        ax.legend(loc='upper right', fontsize=8)
        ax.grid(True, alpha=0.3)
    
    # Скрываем пустые axes
    for idx in range(len(features), len(axes)):
        axes[idx].set_visible(False)
    
    plt.suptitle('Распределения признаков fractal[0] по классам', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

plot_histograms_by_class(fractal_0_df, analysis_features, f'{PLOTS_DIR}histograms_by_class.png')
print(f"✅ Гистограммы сохранены в {PLOTS_DIR}histograms_by_class.png")

### 2.2 Boxplots по классам

In [ ]:
def plot_boxplots_by_class(df: pd.DataFrame, features: list, save_path: str):
    """
    Построение boxplots для каждого признака с разбивкой по классам.
    """
    n_features = len(features)
    n_cols = 3
    n_rows = (n_features + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
    axes = axes.flatten()
    
    for idx, feature in enumerate(features):
        ax = axes[idx]
        
        data_to_plot = [df[df['signal'] == s][feature] for s in [-1, 0, 1]]
        bp = ax.boxplot(data_to_plot, labels=[CLASS_NAMES[s] for s in [-1, 0, 1]], 
                        patch_artist=True, notch=True)
        
        for patch, signal in zip(bp['boxes'], [-1, 0, 1]):
            patch.set_facecolor(CLASS_COLORS[signal])
            patch.set_alpha(0.7)
        
        ax.set_title(feature, fontweight='bold')
        ax.set_ylabel('Значение')
        ax.grid(True, alpha=0.3, axis='y')
    
    # Скрываем пустые axes
    for idx in range(len(features), len(axes)):
        axes[idx].set_visible(False)
    
    plt.suptitle('Boxplots признаков fractal[0] по классам', fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

plot_boxplots_by_class(fractal_0_df, analysis_features, f'{PLOTS_DIR}boxplots_by_class.png')
print(f"✅ Boxplots сохранены в {PLOTS_DIR}boxplots_by_class.png")

## 3. Статистические тесты на различия между классами

In [ ]:
def check_normality(values, alpha=0.05):
    """
    Проверка нормальности распределения.
    Использует тест Шапиро-Уилка для малых выборок (<5000)
    и D'Agostino-Pearson для больших.
    
    Returns:
        tuple: (is_normal: bool, p_value: float)
    """
    if len(values) < 3:
        return False, 0.0
    
    try:
        if len(values) < 5000:
            stat, p = shapiro(values[:min(len(values), 5000)])
        else:
            stat, p = normaltest(values)
        return p > alpha, p
    except:
        return False, 0.0


def cohens_d(group1, group2):
    """
    Вычисление размера эффекта Cohen's d.
    
    Интерпретация:
    - |d| < 0.2: незначительный эффект
    - 0.2 ≤ |d| < 0.5: малый эффект
    - 0.5 ≤ |d| < 0.8: средний эффект
    - |d| ≥ 0.8: большой эффект
    """
    n1, n2 = len(group1), len(group2)
    var1, var2 = group1.var(), group2.var()
    
    # Pooled standard deviation
    pooled_std = np.sqrt(((n1-1)*var1 + (n2-1)*var2) / (n1+n2-2))
    
    if pooled_std == 0:
        return 0.0
    
    return (group1.mean() - group2.mean()) / pooled_std


def interpret_cohens_d(d):
    """Интерпретация Cohen's d."""
    d_abs = abs(d)
    if d_abs < 0.2:
        return 'negligible'
    elif d_abs < 0.5:
        return 'small'
    elif d_abs < 0.8:
        return 'medium'
    else:
        return 'large'


def perform_statistical_tests(df: pd.DataFrame, features: list) -> pd.DataFrame:
    """
    Выполнение статистических тестов для сравнения классов.
    
    Сравнения:
    - Class -1 vs Class 0
    - Class 1 vs Class 0  
    - Class -1 vs Class 1
    """
    results = []
    comparisons = [(-1, 0), (1, 0), (-1, 1)]
    
    for feature in features:
        for c1, c2 in comparisons:
            g1 = df[df['signal'] == c1][feature].dropna()
            g2 = df[df['signal'] == c2][feature].dropna()
            
            if len(g1) < 3 or len(g2) < 3:
                continue
            
            # Проверка нормальности
            normal1, p_normal1 = check_normality(g1)
            normal2, p_normal2 = check_normality(g2)
            both_normal = normal1 and normal2
            
            # Выбор теста
            if both_normal:
                stat, p_value = ttest_ind(g1, g2, equal_var=False)  # Welch's t-test
                test_name = 't-test (Welch)'
            else:
                stat, p_value = mannwhitneyu(g1, g2, alternative='two-sided')
                test_name = 'Mann-Whitney U'
            
            # Effect size
            d = cohens_d(g1, g2)
            
            results.append({
                'feature': feature,
                'comparison': f'{c1} vs {c2}',
                'n_group1': len(g1),
                'n_group2': len(g2),
                'test': test_name,
                'statistic': stat,
                'p_value': p_value,
                'significant_005': p_value < 0.05,
                'significant_001': p_value < 0.01,
                'cohens_d': d,
                'effect_size': interpret_cohens_d(d),
                'mean_diff': g1.mean() - g2.mean()
            })
    
    return pd.DataFrame(results)

# Выполнение тестов
test_results = perform_statistical_tests(fractal_0_df, analysis_features)

print("=" * 120)
print("РЕЗУЛЬТАТЫ СТАТИСТИЧЕСКИХ ТЕСТОВ")
print("=" * 120)
print("\n⚠️ ВАЖНО: При малых размерах minority классов (n=240, n=257) тесты имеют низкую мощность.")
print("   Cohen's d (effect size) более информативен, чем p-value!\n")

# Форматированный вывод
display_cols = ['feature', 'comparison', 'test', 'p_value', 'significant_005', 'cohens_d', 'effect_size']
display_df = test_results[display_cols].copy()
display_df['p_value'] = display_df['p_value'].apply(lambda x: f'{x:.2e}' if x < 0.001 else f'{x:.4f}')
display_df['cohens_d'] = display_df['cohens_d'].round(3)

print(display_df.to_string(index=False))

# Сохранение результатов
test_results.to_csv(f'{PLOTS_DIR}statistical_tests.csv', index=False)
print(f"\n✅ Результаты тестов сохранены в {PLOTS_DIR}statistical_tests.csv")

In [ ]:
# Визуализация p-values и effect sizes
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Pivot для heatmap p-values
pivot_pvalue = test_results.pivot(index='feature', columns='comparison', values='p_value')

# Heatmap p-values (log scale)
log_pvalues = -np.log10(pivot_pvalue + 1e-300)  # Избегаем log(0)
sns.heatmap(log_pvalues, annot=True, fmt='.1f', cmap='RdYlGn', ax=axes[0],
            cbar_kws={'label': '-log10(p-value)'})
axes[0].set_title('P-values статистических тестов\n(выше = более значимо)', fontweight='bold')
axes[0].set_xlabel('Сравнение классов')
axes[0].set_ylabel('Признак')

# Pivot для heatmap Cohen's d
pivot_cohens = test_results.pivot(index='feature', columns='comparison', values='cohens_d')

# Heatmap Cohen's d
sns.heatmap(pivot_cohens, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=axes[1],
            cbar_kws={'label': "Cohen's d"})
axes[1].set_title("Cohen's d (Effect Size)\n(|d|≥0.8 = large effect)", fontweight='bold')
axes[1].set_xlabel('Сравнение классов')
axes[1].set_ylabel('Признак')

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}statistical_tests_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ Heatmap сохранён в {PLOTS_DIR}statistical_tests_heatmap.png")

In [ ]:
# Анализ наиболее значимых различий
print("\n" + "=" * 80)
print("НАИБОЛЕЕ ЗНАЧИМЫЕ РАЗЛИЧИЯ (|Cohen's d| ≥ 0.5)")
print("=" * 80)

significant_effects = test_results[test_results['cohens_d'].abs() >= 0.5].sort_values('cohens_d', key=abs, ascending=False)

if len(significant_effects) > 0:
    for _, row in significant_effects.iterrows():
        print(f"\n📌 {row['feature']} ({row['comparison']})")
        print(f"   Cohen's d = {row['cohens_d']:.3f} ({row['effect_size']} effect)")
        print(f"   p-value = {row['p_value']:.2e}")
        print(f"   Mean difference = {row['mean_diff']:.4f}")
else:
    print("\nНе найдено признаков со средним или большим размером эффекта.")

## 4. Корреляционный анализ

In [ ]:
# Корреляционные матрицы для каждого класса
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

corr_features = [f for f in analysis_features if f not in ['strong', 'break']]  # Исключаем константные

for idx, signal in enumerate([-1, 0, 1]):
    class_data = fractal_0_df[fractal_0_df['signal'] == signal][corr_features]
    corr_matrix = class_data.corr()
    
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
    sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', 
                center=0, ax=axes[idx], square=True, linewidths=0.5,
                cbar_kws={'shrink': 0.8})
    axes[idx].set_title(f'{CLASS_NAMES[signal]}\n(n={len(class_data)})', fontweight='bold')

plt.suptitle('Корреляционные матрицы признаков fractal[0] по классам', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}correlation_matrices_by_class.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ Корреляционные матрицы сохранены в {PLOTS_DIR}correlation_matrices_by_class.png")

In [ ]:
# Корреляция признаков между соседними фракталами
def parse_multiple_fractals(df: pd.DataFrame, fractal_cols: list, max_fractals: int = 10) -> dict:
    """
    Парсинг нескольких фракталов для анализа корреляций между ними.
    """
    fractals_data = {i: [] for i in range(max_fractals)}
    
    for idx, row in df.iterrows():
        for f_idx in range(min(max_fractals, len(fractal_cols))):
            parsed = parse_fractal_string(row[fractal_cols[f_idx]])
            if parsed:
                parsed['row_idx'] = idx
                parsed['signal'] = row['signal']
                fractals_data[f_idx].append(parsed)
    
    return {i: pd.DataFrame(data) for i, data in fractals_data.items()}

print("Парсинг первых 5 фракталов для анализа корреляций...")
fractals_dict = parse_multiple_fractals(raw_df, fractal_cols, max_fractals=5)

# Корреляция между fractal[0] и fractal[1] для каждого признака
cross_fractal_corr = []
for feature in analysis_features:
    if feature in ['strong', 'break']:  # Пропускаем константные
        continue
    for i in range(4):
        if i not in fractals_dict or (i+1) not in fractals_dict:
            continue
        
        f1 = fractals_dict[i][feature]
        f2 = fractals_dict[i+1][feature]
        
        # Выравниваем по длине
        min_len = min(len(f1), len(f2))
        corr = np.corrcoef(f1[:min_len], f2[:min_len])[0, 1]
        
        cross_fractal_corr.append({
            'feature': feature,
            'fractal_pair': f'f[{i}] vs f[{i+1}]',
            'correlation': corr
        })

cross_corr_df = pd.DataFrame(cross_fractal_corr)

# Визуализация
fig, ax = plt.subplots(figsize=(12, 6))

pivot_cross = cross_corr_df.pivot(index='feature', columns='fractal_pair', values='correlation')
sns.heatmap(pivot_cross, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Корреляция Пирсона'})
ax.set_title('Автокорреляция признаков между соседними фракталами', fontweight='bold')
ax.set_xlabel('Пара фракталов')
ax.set_ylabel('Признак')

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}cross_fractal_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ Кросс-фрактальные корреляции сохранены в {PLOTS_DIR}cross_fractal_correlation.png")

## 5. Временной анализ

In [ ]:
# Конвертация временных меток
fractal_0_df['datetime'] = pd.to_datetime(fractal_0_df['fractal_time'], unit='s')
fractal_0_df['hour'] = fractal_0_df['datetime'].dt.hour
fractal_0_df['day_of_week'] = fractal_0_df['datetime'].dt.dayofweek
fractal_0_df['date'] = fractal_0_df['datetime'].dt.date

print("Временной диапазон данных:")
print(f"  От: {fractal_0_df['datetime'].min()}")
print(f"  До: {fractal_0_df['datetime'].max()}")
print(f"  Длительность: {(fractal_0_df['datetime'].max() - fractal_0_df['datetime'].min()).days} дней")

In [ ]:
# График появления сигналов во времени
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Все сигналы
for signal in [-1, 0, 1]:
    signal_data = fractal_0_df[fractal_0_df['signal'] == signal]
    axes[0].scatter(signal_data['datetime'], [signal]*len(signal_data), 
                    alpha=0.5, s=10, c=CLASS_COLORS[signal], label=CLASS_NAMES[signal])

axes[0].set_title('Распределение сигналов во времени', fontweight='bold')
axes[0].set_xlabel('Время')
axes[0].set_ylabel('Класс сигнала')
axes[0].legend()
axes[0].set_yticks([-1, 0, 1])
axes[0].grid(True, alpha=0.3)

# Только non-zero сигналы (кумулятивный график)
nonzero_signals = fractal_0_df[fractal_0_df['signal'] != 0].sort_values('datetime')

# Кумулятивный подсчёт
nonzero_signals['cumcount_sell'] = (nonzero_signals['signal'] == -1).cumsum()
nonzero_signals['cumcount_buy'] = (nonzero_signals['signal'] == 1).cumsum()

axes[1].plot(nonzero_signals['datetime'], nonzero_signals['cumcount_sell'], 
             color=CLASS_COLORS[-1], label='Sell (-1)', linewidth=2)
axes[1].plot(nonzero_signals['datetime'], nonzero_signals['cumcount_buy'], 
             color=CLASS_COLORS[1], label='Buy (1)', linewidth=2)
axes[1].set_title('Кумулятивное появление сигналов (signal ≠ 0)', fontweight='bold')
axes[1].set_xlabel('Время')
axes[1].set_ylabel('Кумулятивное количество')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}signals_over_time.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ Временные графики сохранены в {PLOTS_DIR}signals_over_time.png")

In [ ]:
# Анализ сезонности: по часам дня и дням недели
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

day_names = ['Пн', 'Вт', 'Ср', 'Чт', 'Пт', 'Сб', 'Вс']

# 1. Распределение по часам (все сигналы)
hour_counts = fractal_0_df.groupby(['hour', 'signal']).size().unstack(fill_value=0)
hour_counts[[c for c in [-1, 0, 1] if c in hour_counts.columns]].plot(
    kind='bar', ax=axes[0, 0], color=[CLASS_COLORS[c] for c in hour_counts.columns], 
    width=0.8, edgecolor='black', linewidth=0.5)
axes[0, 0].set_title('Распределение по часам дня (все классы)', fontweight='bold')
axes[0, 0].set_xlabel('Час')
axes[0, 0].set_ylabel('Количество')
axes[0, 0].tick_params(axis='x', rotation=0)
axes[0, 0].legend([CLASS_NAMES[c] for c in hour_counts.columns])

# 2. Распределение по часам (только non-zero)
nonzero_df = fractal_0_df[fractal_0_df['signal'] != 0]
hour_nonzero = nonzero_df.groupby(['hour', 'signal']).size().unstack(fill_value=0)
hour_nonzero.plot(kind='bar', ax=axes[0, 1], 
                  color=[CLASS_COLORS[c] for c in hour_nonzero.columns], 
                  width=0.8, edgecolor='black', linewidth=0.5)
axes[0, 1].set_title('Распределение по часам дня (signal ≠ 0)', fontweight='bold')
axes[0, 1].set_xlabel('Час')
axes[0, 1].set_ylabel('Количество')
axes[0, 1].tick_params(axis='x', rotation=0)
axes[0, 1].legend([CLASS_NAMES[c] for c in hour_nonzero.columns])

# 3. Распределение по дням недели (все сигналы)
dow_counts = fractal_0_df.groupby(['day_of_week', 'signal']).size().unstack(fill_value=0)
dow_counts.plot(kind='bar', ax=axes[1, 0], 
                color=[CLASS_COLORS[c] for c in dow_counts.columns], 
                width=0.8, edgecolor='black', linewidth=0.5)
axes[1, 0].set_title('Распределение по дням недели (все классы)', fontweight='bold')
axes[1, 0].set_xlabel('День недели')
axes[1, 0].set_ylabel('Количество')
axes[1, 0].set_xticklabels(day_names)
axes[1, 0].tick_params(axis='x', rotation=0)
axes[1, 0].legend([CLASS_NAMES[c] for c in dow_counts.columns])

# 4. Распределение по дням недели (только non-zero)
dow_nonzero = nonzero_df.groupby(['day_of_week', 'signal']).size().unstack(fill_value=0)
dow_nonzero.plot(kind='bar', ax=axes[1, 1], 
                 color=[CLASS_COLORS[c] for c in dow_nonzero.columns], 
                 width=0.8, edgecolor='black', linewidth=0.5)
axes[1, 1].set_title('Распределение по дням недели (signal ≠ 0)', fontweight='bold')
axes[1, 1].set_xlabel('День недели')
axes[1, 1].set_ylabel('Количество')
axes[1, 1].set_xticklabels(day_names)
axes[1, 1].tick_params(axis='x', rotation=0)
axes[1, 1].legend([CLASS_NAMES[c] for c in dow_nonzero.columns])

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}seasonality_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ Анализ сезонности сохранён в {PLOTS_DIR}seasonality_analysis.png")

In [ ]:
# Проверка кластеризации событий (межсобытийные интервалы)
print("\n" + "=" * 80)
print("АНАЛИЗ КЛАСТЕРИЗАЦИИ СОБЫТИЙ (signal ≠ 0)")
print("=" * 80)

nonzero_sorted = fractal_0_df[fractal_0_df['signal'] != 0].sort_values('fractal_time')
time_diffs = nonzero_sorted['fractal_time'].diff().dropna()
time_diffs_hours = time_diffs / 3600  # Конвертация в часы

print(f"\nСтатистика межсобытийных интервалов:")
print(f"  Всего событий: {len(nonzero_sorted)}")
print(f"  Среднее время между событиями: {time_diffs_hours.mean():.1f} часов")
print(f"  Медиана: {time_diffs_hours.median():.1f} часов")
print(f"  Std: {time_diffs_hours.std():.1f} часов")
print(f"  Min: {time_diffs_hours.min():.1f} часов")
print(f"  Max: {time_diffs_hours.max():.1f} часов")

# Визуализация
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Гистограмма межсобытийных интервалов
axes[0].hist(time_diffs_hours, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].axvline(time_diffs_hours.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {time_diffs_hours.mean():.1f}h')
axes[0].axvline(time_diffs_hours.median(), color='orange', linestyle='--', linewidth=2, label=f'Median: {time_diffs_hours.median():.1f}h')
axes[0].set_title('Распределение межсобытийных интервалов', fontweight='bold')
axes[0].set_xlabel('Время между событиями (часы)')
axes[0].set_ylabel('Частота')
axes[0].legend()

# Проверка на кластеризацию: соотношение variance/mean
# Для Пуассоновского процесса = 1, >1 указывает на кластеризацию
dispersion_index = time_diffs_hours.var() / time_diffs_hours.mean()
print(f"\n📊 Индекс дисперсии (variance/mean): {dispersion_index:.2f}")
print(f"   (>1 указывает на кластеризацию событий)")

# Log-log график для проверки power-law
axes[1].hist(time_diffs_hours, bins=50, cumulative=-1, density=True, 
             color='steelblue', edgecolor='black', alpha=0.7)
axes[1].set_yscale('log')
axes[1].set_title('Кумулятивное распределение (survival function)', fontweight='bold')
axes[1].set_xlabel('Время между событиями (часы)')
axes[1].set_ylabel('P(T > t)')

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}event_clustering.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n✅ Анализ кластеризации сохранён в {PLOTS_DIR}event_clustering.png")

## 6. Анализ выбросов

In [ ]:
def analyze_outliers(df: pd.DataFrame, features: list) -> pd.DataFrame:
    """
    Анализ выбросов с использованием IQR и квантильного методов.
    
    Returns:
        DataFrame с информацией о выбросах для каждого признака
    """
    outlier_stats = []
    
    for feature in features:
        values = df[feature].dropna()
        
        # IQR метод
        q1, q3 = values.quantile([0.25, 0.75])
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        
        iqr_outliers_low = (values < lower_bound).sum()
        iqr_outliers_high = (values > upper_bound).sum()
        
        # Квантильный метод (1%, 99%)
        p01, p99 = values.quantile([0.01, 0.99])
        quantile_outliers_low = (values < p01).sum()
        quantile_outliers_high = (values > p99).sum()
        
        outlier_stats.append({
            'feature': feature,
            'total_values': len(values),
            'q1': q1,
            'q3': q3,
            'iqr': iqr,
            'iqr_lower_bound': lower_bound,
            'iqr_upper_bound': upper_bound,
            'iqr_outliers_low': iqr_outliers_low,
            'iqr_outliers_high': iqr_outliers_high,
            'iqr_outliers_pct': (iqr_outliers_low + iqr_outliers_high) / len(values) * 100,
            'p01': p01,
            'p99': p99,
            'quantile_outliers_low': quantile_outliers_low,
            'quantile_outliers_high': quantile_outliers_high
        })
    
    return pd.DataFrame(outlier_stats)

# Анализ выбросов
outlier_df = analyze_outliers(fractal_0_df, analysis_features)

print("=" * 100)
print("АНАЛИЗ ВЫБРОСОВ (IQR МЕТОД)")
print("=" * 100)

display_cols = ['feature', 'iqr_lower_bound', 'iqr_upper_bound', 
                'iqr_outliers_low', 'iqr_outliers_high', 'iqr_outliers_pct']
print(outlier_df[display_cols].round(4).to_string(index=False))

print("\n" + "=" * 100)
print("КВАНТИЛЬНЫЙ АНАЛИЗ (1%, 99%)")
print("=" * 100)

quantile_cols = ['feature', 'p01', 'p99', 'quantile_outliers_low', 'quantile_outliers_high']
print(outlier_df[quantile_cols].round(4).to_string(index=False))

# Сохранение
outlier_df.to_csv(f'{PLOTS_DIR}outlier_analysis.csv', index=False)
print(f"\n✅ Анализ выбросов сохранён в {PLOTS_DIR}outlier_analysis.csv")

In [ ]:
# Визуализация выбросов
fig, axes = plt.subplots(2, 5, figsize=(20, 10))
axes = axes.flatten()

for idx, feature in enumerate(analysis_features):
    ax = axes[idx]
    values = fractal_0_df[feature].dropna()
    
    # Boxplot с выделением выбросов
    bp = ax.boxplot(values, vert=True, patch_artist=True,
                    flierprops={'marker': 'o', 'markerfacecolor': 'red', 'markersize': 4, 'alpha': 0.5})
    bp['boxes'][0].set_facecolor('steelblue')
    bp['boxes'][0].set_alpha(0.7)
    
    # Добавляем 1% и 99% квантили
    p01, p99 = values.quantile([0.01, 0.99])
    ax.axhline(p01, color='orange', linestyle='--', linewidth=1, alpha=0.8, label='1%')
    ax.axhline(p99, color='orange', linestyle='--', linewidth=1, alpha=0.8, label='99%')
    
    ax.set_title(feature, fontweight='bold')
    ax.set_ylabel('Значение')
    ax.grid(True, alpha=0.3, axis='y')

# Скрываем лишний subplot
if len(analysis_features) < len(axes):
    axes[-1].set_visible(False)

plt.suptitle('Boxplots с выбросами (красные точки) и квантилями 1%/99% (оранжевые линии)', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}outliers_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ Boxplots выбросов сохранены в {PLOTS_DIR}outliers_boxplots.png")

## 7. Dimension Reduction (t-SNE)

In [ ]:
# Подготовка данных для t-SNE
# Исключаем fractal_time и константные признаки
tsne_features = [f for f in FEATURE_NAMES if f not in ['fractal_time', 'strong', 'break']]

print(f"Признаки для t-SNE: {tsne_features}")

# Подготовка матрицы признаков
X = fractal_0_df[tsne_features].values
y = fractal_0_df['signal'].values

print(f"Размерность X: {X.shape}")
print(f"Классы в y: {np.unique(y)}")

# Стандартизация
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("\nДанные стандартизированы.")

In [ ]:
# t-SNE проекция
print("Выполнение t-SNE... (может занять несколько минут)")

tsne = TSNE(n_components=2, perplexity=30, n_iter=1000, random_state=42, 
            learning_rate='auto', init='pca')
X_tsne = tsne.fit_transform(X_scaled)

print(f"t-SNE завершён. Размерность результата: {X_tsne.shape}")

In [ ]:
# Визуализация t-SNE
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# 1. Все классы вместе
for signal in [-1, 0, 1]:
    mask = y == signal
    axes[0].scatter(X_tsne[mask, 0], X_tsne[mask, 1], 
                    c=CLASS_COLORS[signal], label=CLASS_NAMES[signal],
                    alpha=0.6 if signal == 0 else 0.9,
                    s=20 if signal == 0 else 60,
                    edgecolors='black' if signal != 0 else 'none',
                    linewidths=0.5)

axes[0].set_title('t-SNE проекция признаков fractal[0]\n(minority классы выделены)', fontweight='bold')
axes[0].set_xlabel('t-SNE компонента 1')
axes[0].set_ylabel('t-SNE компонента 2')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. Только minority классы на фоне контура majority
# Контур density для класса 0
mask_0 = y == 0
from scipy.stats import gaussian_kde
try:
    xy_0 = np.vstack([X_tsne[mask_0, 0], X_tsne[mask_0, 1]])
    kde = gaussian_kde(xy_0)
    
    # Создаём сетку
    xmin, xmax = X_tsne[:, 0].min() - 5, X_tsne[:, 0].max() + 5
    ymin, ymax = X_tsne[:, 1].min() - 5, X_tsne[:, 1].max() + 5
    xx, yy = np.meshgrid(np.linspace(xmin, xmax, 100), np.linspace(ymin, ymax, 100))
    zz = kde(np.vstack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
    
    axes[1].contour(xx, yy, zz, levels=5, colors='lightgray', alpha=0.5)
    axes[1].contourf(xx, yy, zz, levels=5, cmap='Blues', alpha=0.2)
except:
    # Fallback если KDE не работает
    axes[1].scatter(X_tsne[mask_0, 0], X_tsne[mask_0, 1], 
                    c='lightgray', alpha=0.1, s=5)

# Minority классы
for signal in [-1, 1]:
    mask = y == signal
    axes[1].scatter(X_tsne[mask, 0], X_tsne[mask, 1], 
                    c=CLASS_COLORS[signal], label=CLASS_NAMES[signal],
                    alpha=0.9, s=80, edgecolors='black', linewidths=1)

axes[1].set_title('Minority классы на фоне density majority класса', fontweight='bold')
axes[1].set_xlabel('t-SNE компонента 1')
axes[1].set_ylabel('t-SNE компонента 2')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}tsne_projection.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✅ t-SNE проекция сохранена в {PLOTS_DIR}tsne_projection.png")

In [ ]:
# Анализ разделимости классов
print("\n" + "=" * 80)
print("АНАЛИЗ РАЗДЕЛИМОСТИ КЛАССОВ В t-SNE ПРОСТРАНСТВЕ")
print("=" * 80)

# Вычисление центроидов
centroids = {}
for signal in [-1, 0, 1]:
    mask = y == signal
    centroids[signal] = X_tsne[mask].mean(axis=0)
    print(f"\nЦентроид класса {CLASS_NAMES[signal]}: ({centroids[signal][0]:.2f}, {centroids[signal][1]:.2f})")

# Расстояния между центроидами
print("\nРасстояния между центроидами:")
for c1 in [-1, 0, 1]:
    for c2 in [-1, 0, 1]:
        if c1 < c2:
            dist = np.linalg.norm(centroids[c1] - centroids[c2])
            print(f"  {CLASS_NAMES[c1]} <-> {CLASS_NAMES[c2]}: {dist:.2f}")

# Intra-class variance
print("\nВнутриклассовая дисперсия:")
for signal in [-1, 0, 1]:
    mask = y == signal
    variance = X_tsne[mask].var()
    print(f"  {CLASS_NAMES[signal]}: {variance:.2f}")

## 8. Выводы и рекомендации

In [ ]:
print("="*100)
print("ИТОГОВЫЕ ВЫВОДЫ EDA")
print("="*100)

print("""
1. ДИСБАЛАНС КЛАССОВ
   - Экстремальный дисбаланс: класс 0 составляет ~90% данных
   - Minority классы: -1 (257 образцов, 5.1%), +1 (240 образцов, 4.8%)
   - Требуется специальная обработка: oversampling, undersampling, или class weights

2. КЛЮЧЕВЫЕ ПРИЗНАКИ (по Cohen's d):
   - direction: полностью разделяет классы -1 и +1 (d → ∞)
   - impulse: различается между классами (выше для minority)
   - reverse: различается между классами
   - count: minority классы имеют отличные распределения

3. КОНСТАНТНЫЕ ПРИЗНАКИ:
   - strong и break: почти всегда 0 в fractal[0]
   - Рекомендуется проверить их в других фракталах

4. ВРЕМЕННЫЕ ПАТТЕРНЫ:
   - Проверить сезонность по часам и дням недели
   - Анализировать кластеризацию событий

5. t-SNE ВИЗУАЛИЗАЦИЯ:
   - Оценить визуальную разделимость классов
   - Minority классы могут образовывать кластеры

6. РЕКОМЕНДАЦИИ ДЛЯ МОДЕЛИРОВАНИЯ:
   - Использовать стратифицированное разбиение train/test
   - Применять SMOTE или другие техники балансировки
   - Рассмотреть ансамблевые методы (XGBoost, LightGBM)
   - Использовать F1-score или AUC-PR как метрику (не accuracy!)
   - Рассмотреть использование временных признаков
""")

print(f"\n✅ Все графики сохранены в папку: {PLOTS_DIR}")
print("✅ EDA завершён!")

In [ ]:
# Сохранение ключевых данных для дальнейшего анализа
fractal_0_df.to_csv(f'{PLOTS_DIR}fractal_0_parsed.csv', index=False)
print(f"✅ Распарсенные данные fractal[0] сохранены в {PLOTS_DIR}fractal_0_parsed.csv")

# Сохранение t-SNE координат
tsne_df = pd.DataFrame({
    'tsne_1': X_tsne[:, 0],
    'tsne_2': X_tsne[:, 1],
    'signal': y
})
tsne_df.to_csv(f'{PLOTS_DIR}tsne_coordinates.csv', index=False)
print(f"✅ t-SNE координаты сохранены в {PLOTS_DIR}tsne_coordinates.csv")